In [153]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch
import matplotlib.dates as mdates
import ast
import folium
from folium.plugins import MarkerCluster
import reverse_geocoder as rg
import re
import pycountry
import os
import numpy as np
import geopandas as gpd
import fiona
import sys
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import ruptures as rpt
from haversine import haversine
import functions as own
from timezonefinder import TimezoneFinder
import zoneinfo
from scipy.stats import gaussian_kde
from scipy.stats import chisquare, kruskal, mannwhitneyu, spearmanr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from libpysal.weights import lat2W
from libpysal.weights import Queen
from libpysal.weights import DistanceBand
from esda.moran import Moran_Local, Moran
from shapely.geometry import box
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
import importlib
from pymannkendall import original_test
from statsmodels.stats.multitest import multipletests

In [50]:
importlib.reload(own)

<module 'functions' from 'c:\\Studium\\X_Masterarbeit\\Data\\Master_Thesis\\Code\\functions.py'>

Lineplot with number of observations per month

In [67]:
df = pd.read_csv("../CWData_clean7.csv")
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")
monthly = df.groupby("year_month").size().reset_index(name="n_obs")
monthly["year_month_dt"] = monthly["year_month"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(16, 9))
ax.scatter(monthly["year_month_dt"], monthly["n_obs"], color="teal", s=20, zorder=3)
ax.plot(monthly["year_month_dt"], monthly["n_obs"], color="teal", linewidth=0.8, alpha=0.4)

ax.fill_between(monthly["year_month_dt"], monthly["n_obs"], alpha=0.1, color="teal")
ax.grid(axis="x", linestyle="--", alpha=0.5)
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.set_axisbelow(True)

plt.title("Total Monthly Observations", fontsize=20)
plt.xlabel("Time", fontsize=17)
plt.ylabel("Number of Observations", fontsize=17)
plt.xticks(rotation=90)

plt.savefig(f"../Products/Lineplot_monthly_Observations.png", dpi=300, bbox_inches="tight")
plt.close()

monthly

C:\Users\yanni\AppData\Local\Temp\ipykernel_39640\1723706834.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

,year_month,n_obs,year_month_dt
0,2017-02,23,2017-02-01
1,2017-03,46,2017-03-01
2,2017-04,34,2017-04-01
3,2017-05,59,2017-05-01
4,2017-06,73,2017-06-01
5,2017-07,55,2017-07-01
6,2017-08,44,2017-08-01
7,2017-09,117,2017-09-01
8,2017-10,104,2017-10-01
9,2017-11,41,2017-11-01


In [68]:
monthly = df.groupby("year_month").size().reset_index(name="n_obs")
monthly["year_month_dt"] = monthly["year_month"].dt.to_timestamp()

result = own.STL_decomposition(monthly, "n_obs", "Monthly Observations", "Observations")

Seasonal strength Monthly Observations: 0.637
Trend strength Monthly Observations:    0.880


In [69]:
trend = result.trend.dropna()
mk = original_test(trend.values)

print(f"Trend: {mk.trend}")
print(f"p-value: {mk.p:.3f}")
print(f"Sen's slope: {mk.slope:.3f} users/month")
print(f"Tau: {mk.Tau:.3f}")

Trend: increasing
p-value: 0.000
Sen's slope: 4.793 users/month
Tau: 0.330


A graphic with three bar charts is created, first the number of observations per hour of day, second the number of observations per month of year, third the number of observations per year

In [128]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
df["hour"] = df["created_at_local"].dt.hour
df["month"] = df["created_at_local"].dt.month
df["year"] = df["created_at_local"].dt.year

# data
hour_counts = df["hour"].value_counts().reindex(range(24), fill_value=0).sort_index()
years_per_month = df.groupby("month")["year"].nunique()
month_counts = (df["month"].value_counts().reindex(range(1,13), fill_value=0).sort_index() / years_per_month).round(0).astype(int)
year_counts = df["year"].value_counts().reindex(range(2017,2027), fill_value=0).sort_index()

fig, axes = plt.subplots(3, 1, figsize=(16, 24))

# Hour
hour_counts.plot(kind="bar", color="teal", ax=axes[0], fontsize=14)
axes[0].set_title("a) Total Observations per Hour of Day (Local Time)", fontsize=22, fontweight="bold", loc="left")
axes[0].set_ylabel("Number of Observations", fontsize=19)
axes[0].set_xlabel("Hour", fontsize=19)
axes[0].set_xticklabels([f"{h:02d}:00-{(h+1)%24:02d}:00" for h in range(24)], rotation=45, ha="right", fontsize=14)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)
axes[0].set_axisbelow(True)

# Month
month_counts.plot(kind="bar", color="teal", ax=axes[1], fontsize=14)
axes[1].set_title("b) Average Observations per Month of Year", fontsize=22, fontweight="bold", loc="left")
axes[1].set_ylabel("Number of Observations", fontsize=19)
axes[1].set_xlabel("Month", fontsize=19)
axes[1].set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], rotation=45, ha="right", fontsize=14)
axes[1].grid(axis="y", linestyle="--", alpha=0.5)
axes[1].set_axisbelow(True)

# Year
year_counts.plot(kind="bar", color="teal", ax=axes[2], fontsize=14)

# extrapolation in 2026
full_years = range(2018, 2026)
april_fracs = []
for y in full_years:
    total = (df["year"] == y).sum()
    until_april = ((df["year"] == y) & (df["month"] <= 4)).sum()
    april_fracs.append(until_april / total)
avg_april_frac = np.mean(april_fracs)
obs_2026_actual = (df["year"] == 2026).sum()
obs_2026_extrap = int(obs_2026_actual / avg_april_frac) - obs_2026_actual

bars = axes[2].patches
last_bar = bars[-1]
axes[2].bar(last_bar.get_x() + last_bar.get_width()/2, obs_2026_extrap,
            bottom=obs_2026_actual, width=last_bar.get_width(),
            color="#90E4C1", hatch="//", edgecolor="teal",
            label=f"2026 extrapolation")
axes[2].legend(fontsize=14)
axes[2].set_title("c) Total Observations per Year", fontsize=22, fontweight="bold", loc="left")
axes[2].set_ylabel("Number of Observations", fontsize=19)
axes[2].set_xlabel("Year", fontsize=19)
axes[2].set_xticklabels(["2017 (from Feb)","2018","2019","2020","2021","2022","2023","2024","2025","2026 (extrapolated)"], rotation=45, ha="right", fontsize=14)
axes[2].grid(axis="y", linestyle="--", alpha=0.5)
axes[2].set_axisbelow(True)

plt.tight_layout()
plt.savefig("../Products/barcharts_hour_month_year.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_39640\3828962262.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

Another double-chart with two stacked bar charts of a) annual and b) monthly new/returning users

In [132]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
df["year"] = df["created_at_local"].dt.year
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")

# data annual
all_years = sorted(df["year"].unique())
seen_users = set()
results = []

for year in all_years:
    year_users = set(df[df["year"] == year]["created_by"].dropna().unique())
    new_users = year_users - seen_users
    returning_users = year_users & seen_users
    results.append({
        "year": year,
        "new_users": len(new_users),
        "returning_users": len(returning_users),
        "total_users": len(year_users)
    })
    seen_users.update(year_users)
result_annual = pd.DataFrame(results)

# data monthly
all_months = sorted(df["year_month"].unique())
seen_users = set()
results = []

for month in all_months:
    month_users = set(df[df["year_month"] == month]["created_by"].dropna().unique())
    new_users = month_users - seen_users
    returning_users = month_users & seen_users
    results.append({
        "year_month": month,
        "new_users": len(new_users),
        "returning_users": len(returning_users),
        "total_users": len(month_users)
    })
    seen_users.update(month_users)
result_monthly = pd.DataFrame(results)
result_monthly["year_month_dt"] = result_monthly["year_month"].dt.to_timestamp()

# Plot
fig, axes = plt.subplots(2, 1, figsize=(16, 16))

# Annual
axes[0].bar(result_annual["year"], result_annual["returning_users"],
            label="Returning Users", color="teal", width=0.6)
axes[0].bar(result_annual["year"], result_annual["new_users"],
            bottom=result_annual["returning_users"],
            label="New Users", color="lightblue", width=0.6)
axes[0].set_title("Annual New vs. Returning Users", fontsize=21, fontweight="bold", loc="left")
axes[0].set_xlabel("Year", fontsize=18)
axes[0].set_ylabel("Number of Users", fontsize=18)
axes[0].set_xticks(result_annual["year"])
axes[0].set_xticklabels(result_annual["year"], rotation=45, ha="right", fontsize=13)
axes[0].legend(fontsize=13)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)
axes[0].set_axisbelow(True)

# Monthly
axes[1].bar(result_monthly["year_month_dt"], result_monthly["returning_users"],
            label="Returning Users", color="teal", width=20)
axes[1].bar(result_monthly["year_month_dt"], result_monthly["new_users"],
            bottom=result_monthly["returning_users"],
            label="New Users", color="lightblue", width=20)
axes[1].set_title("Monthly New vs. Returning Users", fontsize=21, fontweight="bold", loc="left")
axes[1].set_xlabel("Month", fontsize=18)
axes[1].set_ylabel("Number of Users", fontsize=18)
axes[1].tick_params(axis="x", labelsize=13)
axes[1].legend(fontsize=13)
axes[1].grid(axis="y", linestyle="--", alpha=0.5)
axes[1].set_axisbelow(True)
axes[1].set_ylim(0, 200)

plt.tight_layout()
plt.savefig("../Products/new_vs_returning_combined.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_39640\1474056019.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

In [133]:
pd.set_option("display.max_rows", None)
result_monthly

,year_month,new_users,returning_users,total_users,year_month_dt
0,2017-02,2,0,2,2017-02-01
1,2017-03,8,2,10,2017-03-01
2,2017-04,3,4,7,2017-04-01
3,2017-05,15,8,23,2017-05-01
4,2017-06,7,13,20,2017-06-01
5,2017-07,11,9,20,2017-07-01
6,2017-08,4,10,14,2017-08-01
7,2017-09,10,16,26,2017-09-01
8,2017-10,2,15,17,2017-10-01
9,2017-11,3,9,12,2017-11-01


STL decomposition of these data

In [134]:
result1 = own.STL_decomposition(result_monthly, "total_users", "Total Users", "Users")
result2 = own.STL_decomposition(result_monthly, "new_users", "New Users", "Users")
result3 = own.STL_decomposition(result_monthly, "returning_users", "Returning Users", "Users")

Seasonal strength Total Users: 0.779
Trend strength Total Users:    0.840
Seasonal strength New Users: 0.747
Trend strength New Users:    0.528
Seasonal strength Returning Users: 0.602
Trend strength Returning Users:    0.910


Mann-Kendall test (3 times)

In [135]:
trend = result1.trend.dropna()
mk = original_test(trend.values)

print(f"Trend: {mk.trend}")
print(f"p-value: {mk.p:.3f}")
print(f"Sen's slope: {mk.slope:.3f} users/month")
print(f"Tau: {mk.Tau:.3f}")

Trend: increasing
p-value: 0.000
Sen's slope: 0.801 users/month
Tau: 0.555


In [136]:
trend = result2.trend.dropna()
mk = original_test(trend.values)

print(f"Trend: {mk.trend}")
print(f"p-value: {mk.p:.3f}")
print(f"Sen's slope: {mk.slope:.3f} users/month")
print(f"Tau: {mk.Tau:.3f}")

Trend: increasing
p-value: 0.000
Sen's slope: 0.334 users/month
Tau: 0.573


In [137]:
trend = result3.trend.dropna()
mk = original_test(trend.values)

print(f"Trend: {mk.trend}")
print(f"p-value: {mk.p:.3f}")
print(f"Sen's slope: {mk.slope:.3f} users/month")
print(f"Tau: {mk.Tau:.3f}")

Trend: increasing
p-value: 0.000
Sen's slope: 0.471 users/month
Tau: 0.541


Here, monthly maps of observations are created

In [87]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

monthly_counts = (
    df
    .groupby(["year_month", "latitude", "longitude"])
    .size()
    .reset_index(name="count")
)

world = gpd.read_file(f"../Borders/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona").to_crs("+proj=robin")

saved_files = []
for month in monthly_counts["year_month"].unique():
    subset = monthly_counts[monthly_counts["year_month"] == month]
    robin_subset = gpd.GeoDataFrame(
        subset,
        geometry=gpd.points_from_xy(subset["longitude"], subset["latitude"]),
        crs="EPSG:4326"
    ).to_crs("+proj=robin")

    fig, ax = plt.subplots(figsize=(16,8))
    
    # draw world map
    world.plot(ax=ax, color="grey", edgecolor="black", linewidth=0.5)
    
    # plot points
    ax.scatter(
        robin_subset.geometry.x,
        robin_subset.geometry.y,
        s=np.log1p(robin_subset["count"]) * 40, # log for radius
        color="#90E4C1",
        edgecolor="black",
        linewidth=0.3,
        alpha=0.7
    )

    legend_counts = [1, 10, 50, 100]
    legend_handles = [
        plt.scatter([], [], s=np.log1p(c) * 40, color="#90E4C1", edgecolor="black", linewidth=0.7, label=str(c))
        for c in legend_counts
    ]
    legend = ax.legend(
        handles=legend_handles,
        title="Observations",
        loc="lower left",
        frameon=True,
        labelspacing=1.5
    )
    legend.get_title().set_fontsize(15)
    for text in legend.get_texts():
        text.set_fontsize(12)
    
    ax.set_title(f"Observations - {month}", fontsize=18)
    ax.set_axis_off()
    
    filepath = f"../Products/Monthly_Activity_Maps/{month}.png"
    plt.savefig(filepath, dpi=300, bbox_inches="tight")
    plt.close()
    saved_files.append(filepath)

C:\Users\yanni\AppData\Local\Temp\ipykernel_39640\248588565.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 4

In [88]:
own.gif_maker(*saved_files, path="../Products/Monthly_Activity_Maps/Animation", duration=200)

I'm retrieving the date when the first observation is made, the number of observations and the number of unique users per Country.

In [114]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

first_countries = (
    df
    .groupby("Country")
    .agg(
        first_date = ("created_at_local","min"),
        total_observations = ("created_at_local","count"),
        unique_users = ("created_by","nunique")
    )
    .reset_index()
)

pd.set_option("display.max_rows", None)

first_countries_view = (
    first_countries
    .sort_values(by="first_date")
    .assign(first_date=lambda x: x["first_date"].dt.strftime("%d.%m.%Y"))
)
first_countries_view

C:\Users\yanni\AppData\Local\Temp\ipykernel_39640\3679988019.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

,Country,first_date,total_observations,unique_users
80,Switzerland,05.02.2017,23369,1664
5,Austria,04.03.2017,6228,92
30,Germany,19.03.2017,12553,484
28,France,19.03.2017,2282,75
79,Sweden,08.04.2017,242,25
12,Canada,06.05.2017,2170,46
87,United States of America,18.05.2017,1922,143
86,United Kingdom,17.06.2017,5154,155
0,Afghanistan,03.07.2017,1,1
82,Thailand,08.07.2017,2,1


Then, the date of the first observation per country is plotted on a map, this is an overview.

In [96]:
# uses first_countries

world = gpd.read_file(f"../Borders/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")
world = world.rename(columns={"NAME": "Country"})
first_countries["first_date"] = pd.to_datetime(first_countries["first_date"])
first_countries["year"] = first_countries["first_date"].dt.year

bins = [2016, 2017, 2018, 2019, 2020, 2022, 2024, 2026]
labels = ["2017 (Feb-Dec)","2018","2019","2020","2021-2022","2023-2024","2025-2026 (Jan)"]
first_countries["year_bin"] = pd.cut(first_countries["year"], bins=bins, labels=labels)
first_countries["ISO_A3"] = first_countries["Country"].apply(own.get_iso3)

map_df = world.merge(first_countries, left_on="Country", right_on="Country", how="left").to_crs("+proj=robin")

fig, ax = plt.subplots(1, 1, figsize=(16, 8))

cmap = get_cmap("Blues_r", len(labels))
legend_handles = [
    Patch(facecolor=cmap(i / (len(labels) - 1)), edgecolor="black", linewidth=0.5, label=labels[i])
    for i in range(len(labels))
]
legend_handles.append(Patch(facecolor="grey", edgecolor="black", linewidth=0.5, label="No observations"))

map_df.plot(
    column="year_bin",
    cmap="Blues_r",
    linewidth=0.5,
    edgecolor="black",
    missing_kwds={"color": "grey"},
    legend=False,
    categorical=True,
    ax=ax
)

ax.legend(handles=legend_handles, title="Year", title_fontsize=15, loc="lower left")
for text in legend.get_texts():
        text.set_fontsize(12)
ax.set_title("First Observation per Country", fontsize=18)
ax.set_axis_off()

filepath = f"../Products/first_obs_per_Country_total.png"
plt.savefig(filepath, dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_39640\24111754.py:17: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("Blues_r", len(labels))


Then, monthly maps of first observations are created (same colors as the one above).

In [102]:
# uses first_countries

world = gpd.read_file(f"../Borders/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")
world = world.rename(columns={"NAME": "Country"})
first_countries["ISO_A3"] = first_countries["Country"].apply(own.get_iso3)
first_countries["first_date"] = pd.to_datetime(first_countries["first_date"])

bins = [2016, 2017, 2018, 2019, 2020, 2022, 2024, 2026]
labels = ["2017 (Feb-Dec)","2018","2019","2020","2021-2022","2023-2024","2025-2026 (Apr)"]
first_countries["year_bin"] = pd.cut(first_countries["first_date"].dt.year, bins=bins, labels=labels)

# sort months
all_months = pd.period_range(first_countries["first_date"].min(), first_countries["first_date"].max() + pd.offsets.MonthEnd(5), freq="M")

# generate monthly maps
saved_files = []
for month in all_months:
    current = first_countries[first_countries["first_date"] <= month.to_timestamp()]
    if current.empty:
        continue
    
    map_df = world.merge(current, left_on="Country", right_on="Country", how="left").to_crs("+proj=robin")
    
    fig, ax = plt.subplots(1, 1, figsize=(16, 8))
    
    cmap = get_cmap("Blues_r", len(labels))
    legend_handles = [
        Patch(facecolor=cmap(i / (len(labels) - 1)), edgecolor="black", linewidth=0.5, label=labels[i])
        for i in range(len(labels))
    ]
    legend_handles.append(Patch(facecolor="grey", edgecolor="black", linewidth=0.5, label="No observations"))

    map_df.plot(
        column="year_bin",
        cmap="Blues_r",
        linewidth=0.5,
        edgecolor="black",
        missing_kwds={"color": "grey"},
        categorical=True,
        legend=False,
        ax=ax
    )

    ax.legend(handles=legend_handles, title="Year", title_fontsize=15, loc="lower left")
    for text in legend.get_texts():
        text.set_fontsize(12)
    ax.set_title(f"Countries with Observations up to (including) {month-1}", fontsize=18)
    ax.set_axis_off()
    
    filepath = f"../Products/Monthly_first_obs_Maps/{month-1}.png"
    plt.savefig(filepath, dpi=300, bbox_inches="tight")
    plt.close()

    saved_files.append(filepath) # for the following GIF

C:\Users\yanni\AppData\Local\Temp\ipykernel_39640\4153336744.py:26: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("Blues_r", len(labels))
C:\Users\yanni\AppData\Local\Temp\ipykernel_39640\4153336744.py:26: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("Blues_r", len(labels))
C:\Users\yanni\AppData\Local\Temp\ipykernel_39640\4153336744.py:26: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("Blues_r", len(labels))
C:\U

In [103]:
# create GIF

own.gif_maker(*saved_files, path="../Products/Monthly_first_obs_Maps/Animation", duration=200)

Monthly Maps with active/inactive countries.

In [ ]:
world = gpd.read_file("../Borders/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")
world = world.rename(columns={"NAME": "Country"})

df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")

df["ISO_A3"] = df["Country"].apply(own.get_iso3)

pd.set_option("display.max_columns", None)
df.head(3)

countries_ever = set(df["ISO_A3"].dropna().unique())
all_months = pd.period_range(
    df["year_month"].min(),
    df["year_month"].max(),
    freq="M"
)

for month in all_months:
    
    # countries with observations in this month
    current_active = set(
        df.loc[df["year_month"] == month, "ISO_A3"].dropna().unique()
    )
    
    # copy of world map
    map_df = world.copy().to_crs("+proj=robin")
    
    # create status column
    def classify_country(iso):
        if iso in current_active:
            return "Active"
        elif iso in countries_ever:
            return "Inactive"
        else:
            return "Never active"
    
    map_df["status"] = map_df["ADM0_A3"].apply(classify_country)
    
    # plot
    fig, ax = plt.subplots(1, 1, figsize=(16, 8))
    
    categories = ["Active", "Inactive", "Never active"]
    map_df["status"] = pd.Categorical(map_df["status"], categories=categories)

    # define colors
    cmap = mcolors.ListedColormap(["blue", "#df6a91", "grey"])
    color_map = {
        "Active": "blue",
        "Inactive": "#df6a91",
        "Never active": "grey"
    }

    fig, ax = plt.subplots(1, 1, figsize=(16, 8))

    map_df.plot(
        column="status",
        categorical=True,
        cmap=cmap,
        linewidth=0.5,
        edgecolor="black",
        legend=False,
        ax=ax
    )

    legend_handles = [
        Patch(facecolor=color, edgecolor="black", linewidth=0.5, label=label)
        for label, color in color_map.items()
    ]
    
    ax.legend(handles=legend_handles, title="Country Status", title_fontsize=12, loc="lower left")
    ax.set_title(f"Active Countries in {month}", fontsize=15)
    ax.set_axis_off()
    
    plt.savefig(f"../Products/Monthly_active_Countries_Maps/{month}.png",
                dpi=300, bbox_inches="tight")
    plt.close()

Now do the same thing for actual spots, not whole countries.

In [107]:
world = gpd.read_file("../Borders/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")
world = world.rename(columns={"NAME": "Country"})

df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")

df["geometry"] = df.apply(
    lambda row: Point(row["longitude"], row["latitude"]), axis=1
)

gdf = gpd.GeoDataFrame(
    df,
    geometry="geometry",
    crs="EPSG:4326"
).to_crs("+proj=robin")

all_months = pd.period_range(
    gdf["year_month"].min(),
    gdf["year_month"].max(),
    freq="M"
)

for month in all_months:

    fig, ax = plt.subplots(1, 1, figsize=(16, 8))
    
    # map below
    world.to_crs("+proj=robin").plot(
        ax=ax,
        color="grey",
        edgecolor="black",
        linewidth=0.5
    )
    
    # active spots in this month
    active = gdf[gdf["year_month"] == month]
    
    # all other spots
    inactive = gdf[gdf["year_month"] < month]
    
    # first plot inactive ones
    inactive.plot(
        ax=ax,
        color="red",
        markersize=5,
        alpha=0.4
    )
    
    # then active ones
    active.plot(
        ax=ax,
        color="blue",
        markersize=8,
        alpha=0.8
    )

    active_patch = mpatches.Patch(facecolor="blue", edgecolor="black", linewidth=0.5, label="Active")
    inactive_patch = mpatches.Patch(facecolor="red", edgecolor="black", linewidth=0.5, label="Inactive")

    legend = ax.legend(
        handles=[active_patch, inactive_patch],
        title="Spot Status",
        loc="lower left",
        frameon=True
    )

    legend.get_title().set_fontsize(15)
    for text in legend.get_texts():
        text.set_fontsize(12)
    
    ax.set_title(f"Active Observation Spots in {month}", fontsize=18)
    ax.set_axis_off()
    
    plt.savefig(f"../Products/Monthly_active_Spots_Maps/{month}.png",
                dpi=300, bbox_inches="tight")
    plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_39640\4001877958.py:4: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 